In [ ]:
#!/usr/bin/env python3
"""
Сравнение траектории vSLAM с эталонной GPS-траекторией.
1. Читает эталон из .xlsx (метровые координаты)
2. Читает vSLAM из .csv
3. Вычисляет коэффициент масштабирования
4. Поворачивает vSLAM траекторию по направлению эталона
5. Строит обе траектории на одном графике
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. ЧТЕНИЕ ДАННЫХ
# =============================================================================
def load_reference(xlsx_path):
    """Загружает эталонную GPS-траекторию из .xlsx"""
    df = pd.read_excel(xlsx_path)
    print(f"📂 Эталон: {os.path.abspath(xlsx_path)}")
    print(f"   Записей: {len(df)}")
    return df['x_m'].values, df['z_m'].values


def load_vslam(csv_path):
    """Загружает траекторию vSLAM из .csv"""
    df = pd.read_csv(csv_path)
    print(f"📂 vSLAM: {os.path.abspath(csv_path)}")
    print(f"   Записей: {len(df)}")
    return df['x_meters'].values, df['z_meters'].values


# =============================================================================
# 2. МАСШТАБИРОВАНИЕ
# =============================================================================
def compute_scale(x_ref, z_ref, x_vslam, z_vslam):
    """
    Вычисляет коэффициент масштабирования:
    scale = (длина эталона) / (длина vSLAM)
    Длина — расстояние от первой до последней точки траектории.
    """
    dist_ref = np.hypot(x_ref[-1] - x_ref[0], z_ref[-1] - z_ref[0])
    dist_vslam = np.hypot(x_vslam[-1] - x_vslam[0], z_vslam[-1] - z_vslam[0])
    
    if dist_vslam < 1e-6:
        print("️ Длина vSLAM близка к нулю, масштабирование невозможно")
        return 1.0
    
    scale = dist_ref / dist_vslam
    print(f"\n📏 МАСШТАБИРОВАНИЕ:")
    print(f"   Расстояние эталона (старт-финиш): {dist_ref:.2f} м")
    print(f"   Расстояние vSLAM (старт-финиш):   {dist_vslam:.2f} м")
    print(f"   Коэффициент масштабирования:      {scale:.4f}")
    return scale


def apply_scale(x, z, scale):
    """Применяет масштаб к траектории относительно первой точки"""
    x0, z0 = x[0], z[0]
    x_scaled = (x - x0) * scale + x0
    z_scaled = (z - z0) * scale + z0
    return x_scaled, z_scaled


# =============================================================================
# 3. ПОВОРОТ
# =============================================================================
def compute_rotation_angle(x_ref, z_ref, x_vslam, z_vslam):
    """
    Вычисляет угол поворота vSLAM, чтобы направление 
    'первая → последняя' совпало с направлением эталона.
    """
    # Вектор направления эталона
    dx_ref = x_ref[-1] - x_ref[0]
    dz_ref = z_ref[-1] - z_ref[0]
    angle_ref = np.arctan2(dz_ref, dx_ref)
    
    # Вектор направления vSLAM (уже масштабированной)
    dx_vslam = x_vslam[-1] - x_vslam[0]
    dz_vslam = z_vslam[-1] - z_vslam[0]
    angle_vslam = np.arctan2(dz_vslam, dx_vslam)
    
    delta_angle = angle_ref - angle_vslam
    
    print(f"\n🧭 ПОВОРОТ:")
    print(f"   Угол направления эталона: {np.degrees(angle_ref):.2f}°")
    print(f"   Угол направления vSLAM:   {np.degrees(angle_vslam):.2f}°")
    print(f"   Угол поворота:            {np.degrees(delta_angle):.2f}°")
    return delta_angle


def apply_rotation(x, z, angle, pivot_x, pivot_z):
    """
    Поворачивает траекторию вокруг точки (pivot_x, pivot_z) на угол angle.
    """
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    
    # Смещаем к центру вращения
    x_rel = x - pivot_x
    z_rel = z - pivot_z
    
    # Поворачиваем
    x_rot = x_rel * cos_a - z_rel * sin_a
    z_rot = x_rel * sin_a + z_rel * cos_a
    
    # Возвращаем обратно
    return x_rot + pivot_x, z_rot + pivot_z


# =============================================================================
# 4. СМЕЩЕНИЕ (ALIGNMENT)
# =============================================================================
def align_origin(x_vslam, z_vslam, x_ref, z_ref):
    """
    Смещает первую точку vSLAM в первую точку эталона.
    """
    dx = x_ref[0] - x_vslam[0]
    dz = z_ref[0] - z_vslam[0]
    return x_vslam + dx, z_vslam + dz


# =============================================================================
# 5. ГЛАВНАЯ ФУНКЦИЯ ОБРАБОТКИ
# =============================================================================
def align_vslam_to_reference(x_ref, z_ref, x_vslam, z_vslam):
    """
    Полный пайплайн выравнивания vSLAM по эталону:
    1. Масштабирование
    2. Поворот вокруг первой точки
    3. Смещение в начало эталона
    """
    # 1. Масштаб
    scale = compute_scale(x_ref, z_ref, x_vslam, z_vslam)
    x_sc, z_sc = apply_scale(x_vslam, z_vslam, scale)
    
    # 2. Поворот вокруг первой (уже масштабированной) точки
    angle = compute_rotation_angle(x_ref, z_ref, x_sc, z_sc)
    x_rot, z_rot = apply_rotation(x_sc, z_sc, angle, x_sc[0], z_sc[0])
    
    # 3. Смещение в начало эталона
    x_aligned, z_aligned = align_origin(x_rot, z_rot, x_ref, z_ref)
    
    return x_aligned, z_aligned, scale, angle


# =============================================================================
# 6. ВЫЧИСЛЕНИЕ ОШИБКИ
# =============================================================================
def compute_error(x_ref, z_ref, x_aligned, z_aligned):
    """
    Вычисляет метрики ошибки между эталоном и выровненной vSLAM.
    Интерполируем vSLAM в точках эталона (по длине пути).
    """
    # Длины пути для обеих траекторий (накопительная длина)
    s_ref = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_ref), np.diff(z_ref)))])
    s_vslam = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_aligned), np.diff(z_aligned)))])
    
    # Интерполируем vSLAM в точках эталона (по относительной длине пути)
    s_ref_norm = s_ref / s_ref[-1]
    s_vslam_norm = s_vslam / s_vslam[-1]
    
    x_vslam_interp = np.interp(s_ref_norm, s_vslam_norm, x_aligned)
    z_vslam_interp = np.interp(s_ref_norm, s_vslam_norm, z_aligned)
    
    # Ошибки в каждой точке
    errors = np.hypot(x_ref - x_vslam_interp, z_ref - z_vslam_interp)
    
    mae = np.mean(errors)
    rmse = np.sqrt(np.mean(errors ** 2))
    max_err = np.max(errors)
    
    print(f"\n ОШИБКА ПОСЛЕ ВЫРАВНИВАНИЯ (интерполяция по длине пути):")
    print(f"   Средняя ошибка (MAE):  {mae:.2f} м")
    print(f"   Среднеквадратичная (RMSE): {rmse:.2f} м")
    print(f"   Максимальная ошибка:   {max_err:.2f} м")
    
    return errors, mae, rmse, max_err


# =============================================================================
# 7. ВИЗУАЛИЗАЦИЯ
# =============================================================================
def plot_trajectories(x_ref, z_ref, x_vslam_raw, z_vslam_raw,
                     x_vslam_aligned, z_vslam_aligned,
                     output_dir='output'):
    """Строит график сравнения траекторий."""
    os.makedirs(output_dir, exist_ok=True)
    
    fig, ax1 = plt.subplots(figsize=(12, 9))
    
    # === ГРАФИК: Сравнение траекторий ===
    ax1.plot(x_ref, z_ref, 'g--', linewidth=2.5, label='Эталон (GPS)', zorder=3)
    # ax1.plot(x_vslam_raw, z_vslam_raw, 'orange', linewidth=1.5, alpha=0.5, label='vSLAM (сырая)', zorder=1)
    ax1.plot(x_vslam_aligned, z_vslam_aligned, 'b-', linewidth=2.5,
             label='vSLAM (выровненная)', zorder=2)
    
    # Старт и финиш
    ax1.scatter([x_ref[0]], [z_ref[0]], color='lime', s=250, marker='^',
                edgecolors='black', linewidths=2, label='Старт', zorder=5)
    ax1.scatter([x_ref[-1]], [z_ref[-1]], color='red', s=250, marker='v',
                edgecolors='black', linewidths=2, label='Финиш', zorder=5)
    ax1.scatter([x_ref[8]], [z_ref[8]], color='red', s=125, marker='o',
                    edgecolors='black', linewidths=2, label='Светофор', zorder=5)

    
    ax1.set_title('Сравнение траекторий', fontsize=16, fontweight='bold')
    ax1.set_xlabel('X (метры)')
    ax1.set_ylabel('Z (метры)')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.axis('equal')
    ax1.legend(fontsize=11, loc='best')
    
    plt.tight_layout()
    
    out_path = os.path.join(output_dir, 'test_50kmh_2_qhd.png')
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ График сохранён: {os.path.abspath(out_path)}")


# =============================================================================
# MAIN
# =============================================================================
def main():
    # Пути к файлам
    XLSX_FILE = 'output/test_50kmh_2.xlsx'
    CSV_FILE = 'output/merged_trajectory.csv'
    OUTPUT_DIR = 'output'
    
    # 1. Загружаем данные
    x_ref, z_ref = load_reference(XLSX_FILE)
    x_vslam, z_vslam = load_vslam(CSV_FILE)
    
    # 2. Выравниваем vSLAM по эталону
    x_aligned, z_aligned, scale, angle = align_vslam_to_reference(
        x_ref, z_ref, x_vslam, z_vslam
    )
    
    # 3. Вычисляем ошибку (только метрики, без графика)
    errors, mae, rmse, max_err = compute_error(x_ref, z_ref, x_aligned, z_aligned)
    
    # 4. Строим график (без подграфика ошибки)
    plot_trajectories(x_ref, z_ref, x_vslam, z_vslam,
                     x_aligned, z_aligned, OUTPUT_DIR)
    
    # 5. Сохраняем выровненную траекторию в CSV
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    aligned_csv = os.path.join(OUTPUT_DIR, 'vslam_aligned.csv')
    pd.DataFrame({
        'x_meters_raw': x_vslam,
        'z_meters_raw': z_vslam,
        'x_meters_aligned': x_aligned,
        'z_meters_aligned': z_aligned
    }).to_csv(aligned_csv, index=False)
    print(f"✅ Выровненная траектория сохранена: {os.path.abspath(aligned_csv)}")
    
    print(f"\n Готово!")


if __name__ == '__main__':
    main()

📂 Эталон: c:\Users\user\Desktop\регистратор\output\gps_data_with_meters.xlsx
   Записей: 61
📂 vSLAM: c:\Users\user\Desktop\регистратор\output\merged_trajectory.csv
   Записей: 696

📏 МАСШТАБИРОВАНИЕ:
   Расстояние эталона (старт-финиш): 644.75 м
   Расстояние vSLAM (старт-финиш):   680.84 м
   Коэффициент масштабирования:      0.9470

🧭 ПОВОРОТ:
   Угол направления эталона: 134.10°
   Угол направления vSLAM:   85.51°
   Угол поворота:            48.59°

 ОШИБКА ПОСЛЕ ВЫРАВНИВАНИЯ (интерполяция по длине пути):
   Средняя ошибка (MAE):  61.39 м
   Среднеквадратичная (RMSE): 77.09 м
   Максимальная ошибка:   125.77 м

✅ График сохранён: c:\Users\user\Desktop\регистратор\output\test_50kmh_2_qhd.png
✅ Выровненная траектория сохранена: c:\Users\user\Desktop\регистратор\output\vslam_aligned.csv

 Готово!


In [ ]:
#!/usr/bin/env python3
"""
Сравнение траектории vSLAM с эталонной GPS-траекторией.
1. Читает эталон из CSV (lat, lon в последней ячейке каждой строки)
2. Читает vSLAM из merged_trajectory.csv
3. Переводит GPS координаты в метры относительно первой точки
4. Вычисляет коэффициент масштабирования
5. Поворачивает vSLAM траекторию по направлению эталона
6. Строит обе траектории на одном графике
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =============================================================================
# 1. ЧТЕНИЕ ДАННЫХ
# =============================================================================
def load_reference_gps(csv_path):
    """
    Загружает GPS траекторию из CSV.
    Формат: каждая строка содержит одну ячейку с данными, разделёнными запятыми.
    Предпоследний элемент — lat, последний — lon.
    """
    print(f" Чтение GPS эталона: {os.path.abspath(csv_path)}")
    
    lats = []
    lons = []
    
    with open(csv_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Пропускаем первую строку (заголовки)
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        
        # Разделяем по запятой
        parts = line.split(',')
        
        if len(parts) >= 2:
            # Предпоследний — lat, последний — lon
            lat = float(parts[-2])
            lon = float(parts[-1])
            lats.append(lat)
            lons.append(lon)
    
    print(f"   Записей: {len(lats)}")
    print(f"   Первая точка: lat={lats[0]:.6f}, lon={lons[0]:.6f}")
    
    return np.array(lats), np.array(lons)


def load_vslam(csv_path):
    """Загружает траекторию vSLAM из CSV"""
    df = pd.read_csv(csv_path)
    print(f"📂 vSLAM: {os.path.abspath(csv_path)}")
    print(f"   Записей: {len(df)}")
    print(f"   Колонки: {list(df.columns)}")
    
    # Пробуем разные варианты названий колонок
    if 'x_meters' in df.columns and 'z_meters' in df.columns:
        x_col = 'x_meters'
        z_col = 'z_meters'
    elif 'x' in df.columns and 'z' in df.columns:
        x_col = 'x'
        z_col = 'z'
    else:
        raise ValueError(f"Не найдены нужные колонки. Доступные: {list(df.columns)}")
    
    # Явно преобразуем в float (убираем возможные строки)
    x = pd.to_numeric(df[x_col], errors='coerce').values
    z = pd.to_numeric(df[z_col], errors='coerce').values
    
    # Проверяем, что нет NaN
    if np.any(np.isnan(x)) or np.any(np.isnan(z)):
        print(f"⚠️  Найдены нечисловые значения, пропускаем их")
        mask = ~(np.isnan(x) | np.isnan(z))
        x = x[mask]
        z = z[mask]
    
    print(f"   x диапазон: [{x[0]:.2f}, {x[-1]:.2f}]")
    print(f"   z диапазон: [{z[0]:.2f}, {z[-1]:.2f}]")
    
    return x, z


# =============================================================================
# 2. ПЕРЕВОД GPS В МЕТРЫ
# =============================================================================
def gps_to_meters(lats, lons):
    """
    Переводит GPS координаты (lat, lon) в метры относительно первой точки.
    Использует упрощённую проекцию.
    """
    lat_ref = np.radians(lats[0])
    lon_ref = np.radians(lons[0])
    
    # Коэффициенты (приблизительно)
    meters_per_degree_lat = 110540  # метров на градус широты
    meters_per_degree_lon = 111320 * np.cos(lat_ref)  # метров на градус долготы
    
    x = (np.radians(lons) - lon_ref) * meters_per_degree_lon
    z = (np.radians(lats) - lat_ref) * meters_per_degree_lat
    
    return x, z


# =============================================================================
# 3. МАСШТАБИРОВАНИЕ
# =============================================================================
def compute_scale(x_ref, z_ref, x_vslam, z_vslam):
    """
    Вычисляет коэффициент масштабирования:
    scale = (длина эталона) / (длина vSLAM)
    Длина — расстояние от первой до последней точки траектории.
    """
    dist_ref = np.hypot(x_ref[-1] - x_ref[0], z_ref[-1] - z_ref[0])
    dist_vslam = np.hypot(x_vslam[-1] - x_vslam[0], z_vslam[-1] - z_vslam[0])
    
    if dist_vslam < 1e-6:
        print("⚠️ Длина vSLAM близка к нулю, масштабирование невозможно")
        return 1.0
    
    scale = dist_ref / dist_vslam
    print(f"\n📏 МАСШТАБИРОВАНИЕ:")
    print(f"   Расстояние эталона (старт-финиш): {dist_ref:.2f} м")
    print(f"   Расстояние vSLAM (старт-финиш):   {dist_vslam:.2f} м")
    print(f"   Коэффициент масштабирования:      {scale:.4f}")
    return scale


def apply_scale(x, z, scale):
    """Применяет масштаб к траектории относительно первой точки"""
    x0, z0 = x[0], z[0]
    x_scaled = (x - x0) * scale + x0
    z_scaled = (z - z0) * scale + z0
    return x_scaled, z_scaled


# =============================================================================
# 4. ПОВОРОТ
# =============================================================================
def compute_rotation_angle(x_ref, z_ref, x_vslam, z_vslam):
    """
    Вычисляет угол поворота vSLAM, чтобы направление 
    'первая → последняя' совпало с направлением эталона.
    """
    dx_ref = x_ref[-1] - x_ref[0]
    dz_ref = z_ref[-1] - z_ref[0]
    angle_ref = np.arctan2(dz_ref, dx_ref)
    
    dx_vslam = x_vslam[-1] - x_vslam[0]
    dz_vslam = z_vslam[-1] - z_vslam[0]
    angle_vslam = np.arctan2(dz_vslam, dx_vslam)
    
    delta_angle = angle_ref - angle_vslam
    
    print(f"\n🧭 ПОВОРОТ:")
    print(f"   Угол направления эталона: {np.degrees(angle_ref):.2f}°")
    print(f"   Угол направления vSLAM:   {np.degrees(angle_vslam):.2f}°")
    print(f"   Угол поворота:            {np.degrees(delta_angle):.2f}°")
    return delta_angle


def apply_rotation(x, z, angle, pivot_x, pivot_z):
    """Поворачивает траекторию вокруг точки (pivot_x, pivot_z) на угол angle."""
    cos_a = np.cos(angle)
    sin_a = np.sin(angle)
    
    x_rel = x - pivot_x
    z_rel = z - pivot_z
    
    x_rot = x_rel * cos_a - z_rel * sin_a
    z_rot = x_rel * sin_a + z_rel * cos_a
    
    return x_rot + pivot_x, z_rot + pivot_z


# =============================================================================
# 5. СМЕЩЕНИЕ (ALIGNMENT)
# =============================================================================
def align_origin(x_vslam, z_vslam, x_ref, z_ref):
    """Смещает первую точку vSLAM в первую точку эталона."""
    dx = x_ref[0] - x_vslam[0]
    dz = z_ref[0] - z_vslam[0]
    return x_vslam + dx, z_vslam + dz


# =============================================================================
# 6. ГЛАВНАЯ ФУНКЦИЯ ОБРАБОТКИ
# =============================================================================
def align_vslam_to_reference(x_ref, z_ref, x_vslam, z_vslam):
    """
    Полный пайплайн выравнивания vSLAM по эталону:
    1. Масштабирование
    2. Поворот вокруг первой точки
    3. Смещение в начало эталона
    """
    scale = compute_scale(x_ref, z_ref, x_vslam, z_vslam)
    x_sc, z_sc = apply_scale(x_vslam, z_vslam, scale)
    
    angle = compute_rotation_angle(x_ref, z_ref, x_sc, z_sc)
    x_rot, z_rot = apply_rotation(x_sc, z_sc, angle, x_sc[0], z_sc[0])
    
    x_aligned, z_aligned = align_origin(x_rot, z_rot, x_ref, z_ref)
    
    return x_aligned, z_aligned, scale, angle


# =============================================================================
# 7. ВЫЧИСЛЕНИЕ ОШИБКИ
# =============================================================================
def compute_error(x_ref, z_ref, x_aligned, z_aligned):
    """
    Вычисляет метрики ошибки между эталоном и выровненной vSLAM.
    Интерполируем vSLAM в точках эталона (по длине пути).
    """
    s_ref = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_ref), np.diff(z_ref)))])
    s_vslam = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_aligned), np.diff(z_aligned)))])
    
    s_ref_norm = s_ref / s_ref[-1]
    s_vslam_norm = s_vslam / s_vslam[-1]
    
    x_vslam_interp = np.interp(s_ref_norm, s_vslam_norm, x_aligned)
    z_vslam_interp = np.interp(s_ref_norm, s_vslam_norm, z_aligned)
    
    errors = np.hypot(x_ref - x_vslam_interp, z_ref - z_vslam_interp)
    
    mae = np.mean(errors)
    rmse = np.sqrt(np.mean(errors ** 2))
    max_err = np.max(errors)
    
    print(f"\n ОШИБКА ПОСЛЕ ВЫРАВНИВАНИЯ (интерполяция по длине пути):")
    print(f"   Средняя ошибка (MAE):  {mae:.2f} м")
    print(f"   Среднеквадратичная (RMSE): {rmse:.2f} м")
    print(f"   Максимальная ошибка:   {max_err:.2f} м")
    
    return errors, mae, rmse, max_err


# =============================================================================
# 8. ВИЗУАЛИЗАЦИЯ
# =============================================================================
def plot_trajectories(x_ref, z_ref, x_vslam_raw, z_vslam_raw,
                     x_vslam_aligned, z_vslam_aligned,
                     errors, output_dir='output'):
    """Строит график с тремя траекториями и подграфиком ошибки."""
    os.makedirs(output_dir, exist_ok=True)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # === ГРАФИК 1: Сравнение траекторий ===
    ax1.plot(x_ref, z_ref, 'g--', linewidth=2.5, label='Эталон (GPS)', zorder=3)
    ax1.plot(x_vslam_aligned, z_vslam_aligned, 'b-', linewidth=2.5,
             label='vSLAM (выровненная)', zorder=2)
    
    ax1.scatter([x_ref[0]], [z_ref[0]], color='lime', s=250, marker='^',
                edgecolors='black', linewidths=2, label='Старт', zorder=5)
    ax1.scatter([x_ref[-1]], [z_ref[-1]], color='red', s=250, marker='v',
                edgecolors='black', linewidths=2, label='Финиш', zorder=5)
    
    ax1.set_title('Сравнение траекторий', fontsize=16, fontweight='bold')
    ax1.set_xlabel('X (метры)')
    ax1.set_ylabel('Z (метры)')
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.axis('equal')
    ax1.legend(fontsize=11, loc='best')
    
    # === ГРАФИК 2: Ошибка по длине пути ===
    s_ref = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x_ref), np.diff(z_ref)))])
    ax2.plot(s_ref, errors, 'r-', linewidth=2)
    ax2.axhline(np.mean(errors), color='blue', linestyle='--',
                label=f'MAE = {np.mean(errors):.2f} м')
    ax2.fill_between(s_ref, 0, errors, alpha=0.3, color='red')
    
    ax2.set_title('Ошибка vSLAM по длине пути', fontsize=16, fontweight='bold')
    ax2.set_xlabel('Пройденное расстояние (м)')
    ax2.set_ylabel('Ошибка (м)')
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(fontsize=11)
    
    plt.tight_layout()
    
    out_path = os.path.join(output_dir, 'trajectory_comparison.png')
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n✅ График сохранён: {os.path.abspath(out_path)}")


# =============================================================================
# MAIN
# =============================================================================
def main():
    # Пути к файлам
    GPS_CSV_FILE = 'data/server/test_gps_123682_2min.csv'  # ← НОВЫЙ ФАЙЛ
    VSLAM_CSV_FILE = 'output/slam_trajectory_stopped1.csv'
    OUTPUT_DIR = 'output'
    
    # 1. Загружаем GPS данные
    lats, lons = load_reference_gps(GPS_CSV_FILE)
    
    # 2. Переводим GPS в метры относительно первой точки
    x_ref, z_ref = gps_to_meters(lats, lons)
    print(f"   GPS в метрах: x=[{x_ref[0]:.2f}, {x_ref[-1]:.2f}], z=[{z_ref[0]:.2f}, {z_ref[-1]:.2f}]")
    
    # 3. Загружаем vSLAM
    x_vslam, z_vslam = load_vslam(VSLAM_CSV_FILE)
    
    # 4. Выравниваем vSLAM по эталону
    x_aligned, z_aligned, scale, angle = align_vslam_to_reference(
        x_ref, z_ref, x_vslam, z_vslam
    )
    
    # 5. Вычисляем ошибку
    errors, mae, rmse, max_err = compute_error(x_ref, z_ref, x_aligned, z_aligned)
    
    # 6. Строим график
    plot_trajectories(x_ref, z_ref, x_vslam, z_vslam,
                     x_aligned, z_aligned, errors, OUTPUT_DIR)
    
    # 7. Сохраняем выровненную траекторию в CSV
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    aligned_csv = os.path.join(OUTPUT_DIR, 'vslam_aligned.csv')
    pd.DataFrame({
        'x_meters_raw': x_vslam,
        'z_meters_raw': z_vslam,
        'x_meters_aligned': x_aligned,
        'z_meters_aligned': z_aligned
    }).to_csv(aligned_csv, index=False)
    print(f"✅ Выровненная траектория сохранена: {os.path.abspath(aligned_csv)}")
    
    print(f"\n🎉 Готово!")


if __name__ == '__main__':
    main()

 Чтение GPS эталона: c:\Users\user\Desktop\регистратор\data\server\test_gps_123682_2min.csv
   Записей: 49
   Первая точка: lat=54.547849, lon=36.366005
   GPS в метрах: x=[0.00, 5.80], z=[0.00, -7.56]
📂 vSLAM: c:\Users\user\Desktop\регистратор\output\slam_trajectory_stopped1.csv
   Записей: 1023
   Колонки: ['frame', 'x', 'z']
⚠️  Найдены нечисловые значения, пропускаем их
   x диапазон: [-1.72, 985.58]
   z диапазон: [0.97, -135.00]

📏 МАСШТАБИРОВАНИЕ:
   Расстояние эталона (старт-финиш): 9.53 м
   Расстояние vSLAM (старт-финиш):   996.61 м
   Коэффициент масштабирования:      0.0096

🧭 ПОВОРОТ:
   Угол направления эталона: -52.48°
   Угол направления vSLAM:   -7.84°
   Угол поворота:            -44.64°

 ОШИБКА ПОСЛЕ ВЫРАВНИВАНИЯ (интерполяция по длине пути):
   Средняя ошибка (MAE):  0.17 м
   Среднеквадратичная (RMSE): 0.22 м
   Максимальная ошибка:   0.66 м

✅ График сохранён: c:\Users\user\Desktop\регистратор\output\trajectory_comparison.png
✅ Выровненная траектория сохранена: c